### 문제 
- pandas를 이용해서 data 폴더 안에 'rating_train.txt' 파일을 로드 
- 결측치 데이터 제거 
- 중복값 제거 
- 상위의 500개 데이터를 필터링 
- document 데이터와 label 데이터로 나눠준다.(독립 변수, 종속 변수)
- embed 라이브러리 이용하여 토큰화, 벡터화 작업
- 분류 모델은 svc, logistic모델을 이용 (파라미터는 기본값)
- 벡터화 방법 2가지를 이용하고 모델 2개를 이용하여 성능 평가 
- 최적의 모델을 이용하여 하위의 200개의 데이터의 예측을 하고 분류 레포트를 생성 

In [1]:
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [3]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [5]:
df.dropna(inplace=True)

In [7]:
# 중복 데이터를 제거하는 함수 
df.drop_duplicates('document', inplace=True)

In [8]:
df2 = df[:500]

In [9]:
df2.info()

<class 'pandas.DataFrame'>
Index: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        500 non-null    int64
 1   document  500 non-null    str  
 2   label     500 non-null    int64
dtypes: int64(2), str(1)
memory usage: 15.6 KB


In [10]:
df2['label'].value_counts()

label
1    258
0    242
Name: count, dtype: int64

In [11]:
X = df2['document'].values
y = df2['label'].values

In [12]:
import embed

In [13]:
# 토큰화 된 데이터를 되돌려주고 module 안에 wv, idf_weight 변수를 생성
tokens = embed.global_vari_set(X)

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
logi_wv = LogisticRegression(random_state = 42)
logi_idf = LogisticRegression(random_state = 42)
svc_wv = SVC(random_state=42)
svc_idf = SVC(random_state = 42)

In [15]:
# tokens을 이용하여 벡터화 임베딩 작업 
X_wv = []
X_idf = []
for token in tokens:
    vec_wv = embed.sent_embed_mean(token)
    vec_idf = embed.sent_embed_wv_idf(token)
    X_wv.append(vec_wv)
    X_idf.append(vec_idf)

In [16]:
print(embed.run_model(X_wv, y, logi_wv))
print(embed.run_model(X_wv, y, svc_wv))
print(embed.run_model(X_idf, y, logi_idf))
print(embed.run_model(X_idf, y, svc_idf))

              precision    recall  f1-score   support

           0       0.62      0.61      0.61        51
           1       0.60      0.61      0.61        49

    accuracy                           0.61       100
   macro avg       0.61      0.61      0.61       100
weighted avg       0.61      0.61      0.61       100

              precision    recall  f1-score   support

           0       0.64      0.64      0.64        50
           1       0.64      0.64      0.64        50

    accuracy                           0.64       100
   macro avg       0.64      0.64      0.64       100
weighted avg       0.64      0.64      0.64       100

              precision    recall  f1-score   support

           0       0.60      0.61      0.61        49
           1       0.62      0.61      0.61        51

    accuracy                           0.61       100
   macro avg       0.61      0.61      0.61       100
weighted avg       0.61      0.61      0.61       100

              preci

In [17]:
# 최적의 모델(svc_idf)을 이용하여 예측 
X_test = df2.tail(100)['document'].values
y_test = df2.tail(100)['label'].values

In [19]:
pred = embed.predict_sentence_list(X_test, svc_idf, 'idf')

In [ ]:
pred_df = pd.DataFrame(pred, columns = ['sentence', 'label'])

# print(classification_report(pred_df['label'].values, y_test))
display(pred_df)

In [22]:
pred_y = pred_df['label'].map({
    '부정' : 0, 
    '긍정' : 1
})
print(classification_report(pred_y, y_test))

              precision    recall  f1-score   support

           0       0.73      0.75      0.74        40
           1       0.83      0.82      0.82        60

    accuracy                           0.79       100
   macro avg       0.78      0.78      0.78       100
weighted avg       0.79      0.79      0.79       100

